In [69]:
import pandas as pd
import numpy as np
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

In [70]:
ctx = snowflake.connector.connect(
    user='BEEUTE',
    password="Chichi7muanya*",
    account='PXMKPZA-TC44237', # e.g., xy12345.us-east-2
    warehouse='COMPUTE_WH',
    database='BI_CONSUMPTION',
    schema='RAW'
) 

In [71]:
data = "36100609.csv"

consumption = pd.read_csv(data)

consumption.head(5)

,REF_DATE,GEO,DGUID,Estimates,Consumption category,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2008,Canada,2016A000011124,Household final consumption expenditure,Total consumption,Dollars,81,millions,6,v1039061489,1.1.1,878509.0,NaN,NaN,NaN,1
1,2008,Canada,2016A000011124,Household final consumption expenditure,Food and non-alcoholic beverages,Dollars,81,millions,6,v1039061490,1.1.2,79281.0,NaN,NaN,NaN,1
2,2008,Canada,2016A000011124,Household final consumption expenditure,Alcoholic beverages and tobacco,Dollars,81,millions,6,v1039061491,1.1.3,35323.0,NaN,NaN,NaN,1
3,2008,Canada,2016A000011124,Household final consumption expenditure,Clothing and footwear,Dollars,81,millions,6,v1039061492,1.1.4,37433.0,NaN,NaN,NaN,1
4,2008,Canada,2016A000011124,Household final consumption expenditure,"Housing, water, electricity, gas and other fuels",Dollars,81,millions,6,v1039061493,1.1.5,200727.0,NaN,NaN,NaN,1


### SELECTING THE COLUMNS I AM USING

In [72]:
consumption = consumption.drop(['UOM_ID', 'SCALAR_ID', 'VECTOR', 'COORDINATE', "STATUS", 'SYMBOL', 'TERMINATED', 'DECIMALS'], axis=1)

consumption.head()

,REF_DATE,GEO,DGUID,Estimates,Consumption category,UOM,SCALAR_FACTOR,VALUE
0,2008,Canada,2016A000011124,Household final consumption expenditure,Total consumption,Dollars,millions,878509.0
1,2008,Canada,2016A000011124,Household final consumption expenditure,Food and non-alcoholic beverages,Dollars,millions,79281.0
2,2008,Canada,2016A000011124,Household final consumption expenditure,Alcoholic beverages and tobacco,Dollars,millions,35323.0
3,2008,Canada,2016A000011124,Household final consumption expenditure,Clothing and footwear,Dollars,millions,37433.0
4,2008,Canada,2016A000011124,Household final consumption expenditure,"Housing, water, electricity, gas and other fuels",Dollars,millions,200727.0


In [73]:
#CHECKING INFORMATION ON THE COLUMNS

consumption.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11880 entries, 0 to 11879
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   REF_DATE              11880 non-null  int64  
 1   GEO                   11880 non-null  object 
 2   DGUID                 11088 non-null  object 
 3   Estimates             11880 non-null  object 
 4   Consumption category  11880 non-null  object 
 5   UOM                   11880 non-null  object 
 6   SCALAR_FACTOR         11880 non-null  object 
 7   VALUE                 11264 non-null  float64
dtypes: float64(1), int64(1), object(6)
memory usage: 742.6+ KB


In [74]:
consumption.describe()

,REF_DATE,VALUE
count,11880.000000,1.126400e+04
mean,2016.500000,1.918810e+04
std,5.188346,9.367904e+04
min,2008.000000,-1.609800e+04
25%,2012.000000,9.790000e+01
50%,2016.500000,1.135300e+03
75%,2021.000000,7.787075e+03
max,2025.000000,2.289076e+06


In [75]:
#CHECKING NULL VALUES
consumption.isnull().sum()

REF_DATE                  0
GEO                       0
DGUID                   792
Estimates                 0
Consumption category      0
UOM                       0
SCALAR_FACTOR             0
VALUE                   616
dtype: int64

In [76]:
#CHECKING FOR DUPLICATES
consumption.duplicated().sum()

0

### CLEANING 

In [77]:
#FIXING NULL VALUES

consumption = consumption.dropna()

consumption.isnull().sum()

REF_DATE                0
GEO                     0
DGUID                   0
Estimates               0
Consumption category    0
UOM                     0
SCALAR_FACTOR           0
VALUE                   0
dtype: int64

In [78]:
#RENAMING COLUMNS
#Note I am making all the columns name upper case because the Data warehouse (SNOWFLAKE) I am working with requires that.

consumption.columns = (consumption.columns.str.upper().str.replace(" ", "_"))

consumption


,REF_DATE,GEO,DGUID,ESTIMATES,CONSUMPTION_CATEGORY,UOM,SCALAR_FACTOR,VALUE
0,2008,Canada,2016A000011124,Household final consumption expenditure,Total consumption,Dollars,millions,878509.0
1,2008,Canada,2016A000011124,Household final consumption expenditure,Food and non-alcoholic beverages,Dollars,millions,79281.0
2,2008,Canada,2016A000011124,Household final consumption expenditure,Alcoholic beverages and tobacco,Dollars,millions,35323.0
3,2008,Canada,2016A000011124,Household final consumption expenditure,Clothing and footwear,Dollars,millions,37433.0
4,2008,Canada,2016A000011124,Household final consumption expenditure,"Housing, water, electricity, gas and other fuels",Dollars,millions,200727.0
...,...,...,...,...,...,...,...,...
11259,2025,Canada,2016A000011124,Household actual final consumption,"Food, beverage and accommodation services",Dollars,millions,129124.0
11260,2025,Canada,2016A000011124,Household actual final consumption,Insurance and financial services,Dollars,millions,160361.0
11261,2025,Canada,2016A000011124,Household actual final consumption,Miscellaneous goods and services (including so...,Dollars,millions,133207.1
11262,2025,Canada,2016A000011124,Household actual final consumption,Net expenditure abroad,Dollars,millions,-16098.0


In [79]:
consumption["REF_DATE"] = pd.to_datetime(consumption["REF_DATE"].astype(str) + "-01-01")

consumption

,REF_DATE,GEO,DGUID,ESTIMATES,CONSUMPTION_CATEGORY,UOM,SCALAR_FACTOR,VALUE
0,2008-01-01,Canada,2016A000011124,Household final consumption expenditure,Total consumption,Dollars,millions,878509.0
1,2008-01-01,Canada,2016A000011124,Household final consumption expenditure,Food and non-alcoholic beverages,Dollars,millions,79281.0
2,2008-01-01,Canada,2016A000011124,Household final consumption expenditure,Alcoholic beverages and tobacco,Dollars,millions,35323.0
3,2008-01-01,Canada,2016A000011124,Household final consumption expenditure,Clothing and footwear,Dollars,millions,37433.0
4,2008-01-01,Canada,2016A000011124,Household final consumption expenditure,"Housing, water, electricity, gas and other fuels",Dollars,millions,200727.0
...,...,...,...,...,...,...,...,...
11259,2025-01-01,Canada,2016A000011124,Household actual final consumption,"Food, beverage and accommodation services",Dollars,millions,129124.0
11260,2025-01-01,Canada,2016A000011124,Household actual final consumption,Insurance and financial services,Dollars,millions,160361.0
11261,2025-01-01,Canada,2016A000011124,Household actual final consumption,Miscellaneous goods and services (including so...,Dollars,millions,133207.1
11262,2025-01-01,Canada,2016A000011124,Household actual final consumption,Net expenditure abroad,Dollars,millions,-16098.0


### FEATURE ENGINEERNG

In [80]:
#CREATING A YEAR COLUMN

consumption["YEAR"] = consumption["REF_DATE"].dt.year
consumption

,REF_DATE,GEO,DGUID,ESTIMATES,CONSUMPTION_CATEGORY,UOM,SCALAR_FACTOR,VALUE,YEAR
0,2008-01-01,Canada,2016A000011124,Household final consumption expenditure,Total consumption,Dollars,millions,878509.0,2008
1,2008-01-01,Canada,2016A000011124,Household final consumption expenditure,Food and non-alcoholic beverages,Dollars,millions,79281.0,2008
2,2008-01-01,Canada,2016A000011124,Household final consumption expenditure,Alcoholic beverages and tobacco,Dollars,millions,35323.0,2008
3,2008-01-01,Canada,2016A000011124,Household final consumption expenditure,Clothing and footwear,Dollars,millions,37433.0,2008
4,2008-01-01,Canada,2016A000011124,Household final consumption expenditure,"Housing, water, electricity, gas and other fuels",Dollars,millions,200727.0,2008
...,...,...,...,...,...,...,...,...,...
11259,2025-01-01,Canada,2016A000011124,Household actual final consumption,"Food, beverage and accommodation services",Dollars,millions,129124.0,2025
11260,2025-01-01,Canada,2016A000011124,Household actual final consumption,Insurance and financial services,Dollars,millions,160361.0,2025
11261,2025-01-01,Canada,2016A000011124,Household actual final consumption,Miscellaneous goods and services (including so...,Dollars,millions,133207.1,2025
11262,2025-01-01,Canada,2016A000011124,Household actual final consumption,Net expenditure abroad,Dollars,millions,-16098.0,2025


In [81]:
#REMOVING ROWS WITH CANADA OR TOTAL CONSUMPTION BECAUSE WE ONLY NEED THE PROVINCES
consumption = consumption[consumption["GEO"] != "Canada"]
consumption = consumption[consumption["CONSUMPTION_CATEGORY"] != "Total consumption"]
consumption

,REF_DATE,GEO,DGUID,ESTIMATES,CONSUMPTION_CATEGORY,UOM,SCALAR_FACTOR,VALUE,YEAR
45,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Food and non-alcoholic beverages,Dollars,millions,1218.0,2008
46,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Alcoholic beverages and tobacco,Dollars,millions,698.6,2008
47,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Clothing and footwear,Dollars,millions,540.5,2008
48,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,"Housing, water, electricity, gas and other fuels",Dollars,millions,2623.7,2008
49,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,"Furnishings, household equipment and other goo...",Dollars,millions,730.9,2008
...,...,...,...,...,...,...,...,...,...
11171,2024-01-01,Nunavut,2016A000262,Household actual final consumption,"Food, beverage and accommodation services",Dollars,millions,48.0,2024
11172,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Insurance and financial services,Dollars,millions,35.5,2024
11173,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Miscellaneous goods and services (including so...,Dollars,millions,165.7,2024
11174,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Net expenditure abroad,Dollars,millions,-2.3,2024


In [82]:
#CREATING PREVIOUS YEAR VALUE CALCULATION
key_cols = ["YEAR", "GEO", "ESTIMATES", "CONSUMPTION_CATEGORY"]

prev_year = consumption[key_cols + ["VALUE"]].copy()
prev_year["YEAR"] = prev_year["YEAR"] + 1
prev_year = prev_year.rename(columns={"VALUE": "PREVIOUS_YEAR_VALUE"})

consumption = consumption.merge(prev_year, on=key_cols, how="left")

consumption

,REF_DATE,GEO,DGUID,ESTIMATES,CONSUMPTION_CATEGORY,UOM,SCALAR_FACTOR,VALUE,YEAR,PREVIOUS_YEAR_VALUE
0,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Food and non-alcoholic beverages,Dollars,millions,1218.0,2008,NaN
1,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Alcoholic beverages and tobacco,Dollars,millions,698.6,2008,NaN
2,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Clothing and footwear,Dollars,millions,540.5,2008,NaN
3,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,"Housing, water, electricity, gas and other fuels",Dollars,millions,2623.7,2008,NaN
4,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,"Furnishings, household equipment and other goo...",Dollars,millions,730.9,2008,NaN
...,...,...,...,...,...,...,...,...,...,...
8835,2024-01-01,Nunavut,2016A000262,Household actual final consumption,"Food, beverage and accommodation services",Dollars,millions,48.0,2024,45.0
8836,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Insurance and financial services,Dollars,millions,35.5,2024,32.5
8837,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Miscellaneous goods and services (including so...,Dollars,millions,165.7,2024,156.7
8838,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Net expenditure abroad,Dollars,millions,-2.3,2024,-0.5


In [83]:
#Year Over Year Growth Formular

consumption['YEAR_OVER_YEAR_GROWTH'] = ((consumption["VALUE"] - consumption["PREVIOUS_YEAR_VALUE"])/ consumption["PREVIOUS_YEAR_VALUE"]) * 100

consumption

,REF_DATE,GEO,DGUID,ESTIMATES,CONSUMPTION_CATEGORY,UOM,SCALAR_FACTOR,VALUE,YEAR,PREVIOUS_YEAR_VALUE,YEAR_OVER_YEAR_GROWTH
0,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Food and non-alcoholic beverages,Dollars,millions,1218.0,2008,NaN,NaN
1,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Alcoholic beverages and tobacco,Dollars,millions,698.6,2008,NaN,NaN
2,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Clothing and footwear,Dollars,millions,540.5,2008,NaN,NaN
3,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,"Housing, water, electricity, gas and other fuels",Dollars,millions,2623.7,2008,NaN,NaN
4,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,"Furnishings, household equipment and other goo...",Dollars,millions,730.9,2008,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
8835,2024-01-01,Nunavut,2016A000262,Household actual final consumption,"Food, beverage and accommodation services",Dollars,millions,48.0,2024,45.0,6.666667
8836,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Insurance and financial services,Dollars,millions,35.5,2024,32.5,9.230769
8837,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Miscellaneous goods and services (including so...,Dollars,millions,165.7,2024,156.7,5.743459
8838,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Net expenditure abroad,Dollars,millions,-2.3,2024,-0.5,360.000000


In [84]:
consumption = consumption.fillna(0)

consumption

,REF_DATE,GEO,DGUID,ESTIMATES,CONSUMPTION_CATEGORY,UOM,SCALAR_FACTOR,VALUE,YEAR,PREVIOUS_YEAR_VALUE,YEAR_OVER_YEAR_GROWTH
0,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Food and non-alcoholic beverages,Dollars,millions,1218.0,2008,0.0,0.000000
1,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Alcoholic beverages and tobacco,Dollars,millions,698.6,2008,0.0,0.000000
2,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,Clothing and footwear,Dollars,millions,540.5,2008,0.0,0.000000
3,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,"Housing, water, electricity, gas and other fuels",Dollars,millions,2623.7,2008,0.0,0.000000
4,2008-01-01,Newfoundland and Labrador,2016A000210,Household final consumption expenditure,"Furnishings, household equipment and other goo...",Dollars,millions,730.9,2008,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
8835,2024-01-01,Nunavut,2016A000262,Household actual final consumption,"Food, beverage and accommodation services",Dollars,millions,48.0,2024,45.0,6.666667
8836,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Insurance and financial services,Dollars,millions,35.5,2024,32.5,9.230769
8837,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Miscellaneous goods and services (including so...,Dollars,millions,165.7,2024,156.7,5.743459
8838,2024-01-01,Nunavut,2016A000262,Household actual final consumption,Net expenditure abroad,Dollars,millions,-2.3,2024,-0.5,360.000000


In [86]:
conn = ctx
write_pandas(conn, consumption, "BI_CONSUMPTION_RAW", schema="RAW")

(True,
 1,
 8840,
 [('snowpark_temp_stage_hcrgo6val8/file0.txt',
   'LOADED',
   8840,
   8840,
   1,
   0,
   None,
   None,
   None,
   None)])